# 07. 토스광고 CTR 이종 모델 OOF 블렌딩, 50:50 WLL 확률 보정 및 최종 제출

이 노트북은 GBDT 모델군(`LightGBM`, `CatBoost`, `XGBoost`)과 시퀀스 딥러닝 모델(`SeqDCNv2CTRModel`)이 생성한 표준 예측 아티팩트를 결합해 **최종 리더보드 점수 `0.5 * AP + 0.5 * (1 / (1 + WLL))`을 극대화**하고 제출 무결성을 검증합니다.

### 핵심 설계 원칙 (누수 제로 & `AP`/`WLL` 동시 최적화)
1. **엄격한 정합성 검증**: `assert_oof_alignment` 및 `assert_test_alignment`로 모든 모델의 `row_id`, `fold_id`, `clicked`, `ID` 순서 100% 일치를 확인합니다.
2. **Out-of-Fold Log-odds 블렌딩**: `OOFLogitBlender`로 각 폴드에서 학습(`fold != k`)한 비음수 가중치로 검증 폴드(`fold == k`)를 결합해 가중치 과적합을 차단합니다.
3. **단조 `50:50 WLL` 보정**: `AffineLogitCalibrator` (`a > 0`)를 `OutOfFoldCalibrator`로 피팅해 **`AP`를 100% 보존하면서 `50:50 WLL`을 정밀 보정**하고, 선택적으로 `GroupBiasShrinkageCalibrator`의 OOF 개선 여부를 검증합니다.
4. **제출 무결성 & 실시간 서빙 검증**: `1,527,298`행, `ID` 순서, `NaN/Inf=0`, `(0, 1)` 확률 범위를 자동 검증하고 `RealtimeCTRPredictor` 서빙 구성을 확인합니다.

In [ ]:
from google.colab import drive
import gc
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import os as _os
if not _os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# TODO: Google Drive 내 프로젝트 루트 경로에 맞게 수정하세요.
PROJECT_ROOT = Path('/content/drive/MyDrive/toss_ctr')
if (PROJECT_ROOT / 'src').exists() and str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if (PROJECT_ROOT / 'pyproject.toml').exists():
    os.chdir(PROJECT_ROOT)
    %pip install -e . --no-deps --quiet

from toss_ctr.evaluation.metrics import evaluate_predictions, weighted_logloss_50_50
from toss_ctr.inference import (
    OOFLogitBlender,
    OfflineFeatureStore,
    RealtimeCTRPredictor,
    assert_oof_alignment,
    assert_test_alignment,
    generate_submission,
    load_prediction_artifacts,
    load_yaml_with_overrides,
    verify_submission_df,
)
from toss_ctr.inference.blender import _id_arrays_equal
from toss_ctr.models.calibration import (
    AffineLogitCalibrator,
    GroupBiasShrinkageCalibrator,
    OutOfFoldCalibrator,
    logit_to_prob,
)

cfg_path = PROJECT_ROOT / 'configs' / 'ensemble_wll50.yaml'
cfg = load_yaml_with_overrides(cfg_path)
paths_cfg = cfg['paths']
models_cfg = cfg['models']
blend_cfg = cfg['blender']
cal_cfg = cfg['calibration']
sub_cfg = cfg['submission']

print('Ensemble config loaded:', cfg_path)
print('Candidate models:', models_cfg['candidates'])

### [해석 1] Colab 환경 구성 및 앙상블·보정 설정 로드
- Google Drive 마운트 및 `toss_ctr` 패키지 경로 설정 후 `configs/ensemble_wll50.yaml`을 로드했습니다.
- 후보 모델 목록(`lgbm_baseline`, `lgbm_fe_tuned`, `catboost_fe_tuned`, `xgb_fe_tuned`, `deep_seq_dcn`)과 OOF 블렌더·캘리브레이션·제출 검증 파라미터가 정상 준비되었습니다.

In [ ]:
preds_dir = Path(paths_cfg['preds_dir'])
sample_sub_path = Path(paths_cfg['sample_submission'])
id_col = sub_cfg['id_col']

sample_ids_df = None
reference_ids = None
if sample_sub_path.exists():
    sample_ids_df = pd.read_csv(sample_sub_path, usecols=[id_col], dtype={id_col: str})
    probe_names = list(models_cfg['candidates']) if models_cfg.get('candidates') else [
        p.stem[len('test_'):] for p in sorted(preds_dir.glob('test_*.parquet'))
    ]
    for m_name in probe_names:
        test_p = preds_dir / f'test_{m_name}.parquet'
        if test_p.exists():
            pf_test = pq.ParquetFile(test_p)
            if id_col in pf_test.schema_arrow.names and pa.types.is_integer(pf_test.schema_arrow.field(id_col).type):
                sample_ids_df[id_col] = sample_ids_df[id_col].astype(np.int64)
            break
    reference_ids = sample_ids_df[id_col].to_numpy()
    print(f'sample_submission rows: {len(sample_ids_df):,}')
else:
    print('sample_submission.csv 미존재 -> 모델 간 test ID 상호 일치만 검증합니다.')

oof_dfs, test_dfs = load_prediction_artifacts(
    preds_dir=preds_dir,
    model_names=models_cfg['candidates'],
    require_test=bool(models_cfg.get('require_test_preds', True)),
    skip_missing=bool(models_cfg.get('auto_select_existing', True)),
    reference_test_ids=reference_ids,
)

row_ids, fold_ids, y_all = assert_oof_alignment(oof_dfs)
if test_dfs:
    assert_test_alignment(test_dfs, reference_ids=reference_ids)

print(f'Loaded & aligned models ({len(oof_dfs)}): {list(oof_dfs.keys())}')
print(f'OOF rows: {len(row_ids):,} | Positive CTR: {y_all.mean():.4%} | Folds: {np.unique(fold_ids).tolist()}')

### [해석 2] 표준 OOF / Test 예측 아티팩트 로드 및 정합성 검증
- `preds_dir`에서 존재하는 모델들의 `oof_*.parquet`과 `test_*.parquet`을 자동 탐지해 로드했습니다.
- 모든 모델 간 `row_id` 오름차순, `fold_id` 폴드 배정, `clicked` 타겟, 그리고 `sample_submission.csv`와의 `ID` 순서가 100% 일치함을 확인해 이후 블렌딩 과정의 정렬 오류를 원천 차단했습니다.

In [ ]:
folds_path = Path(paths_cfg['folds_path'])
oot_mask = None
if folds_path.exists():
    folds_meta = pq.read_table(folds_path, columns=['row_id', 'is_oot_val']).to_pandas()
    if len(folds_meta) == len(row_ids) and 'is_oot_val' in folds_meta.columns:
        oot_mask = folds_meta['is_oot_val'].to_numpy(dtype=bool)

single_rows = []
logit_frame = {}
for name, df in oof_dfs.items():
    probs = df['pred_prob'].to_numpy(dtype=np.float64)
    ev = evaluate_predictions(y_all, probs)
    rec = {
        'model': name,
        'ap': ev.ap,
        'wll': ev.wll,
        'pos_logloss': ev.pos_logloss,
        'neg_logloss': ev.neg_logloss,
        'score': ev.score,
        'pred_mean': float(np.mean(probs)),
    }
    if oot_mask is not None and np.any(oot_mask) and np.unique(y_all[oot_mask]).size == 2:
        oot_ev = evaluate_predictions(y_all[oot_mask], probs[oot_mask])
        rec['oot_ap'] = oot_ev.ap
        rec['oot_wll'] = oot_ev.wll
        rec['oot_score'] = oot_ev.score
    single_rows.append(rec)
    logit_frame[name] = df['raw_logit'].to_numpy(dtype=np.float64)

single_df = pd.DataFrame(single_rows).sort_values('score', ascending=False).reset_index(drop=True)
display(single_df)

corr_df = pd.DataFrame(logit_frame).corr(method='pearson')
print('\n=== Raw Logit Pearson Correlation Matrix ===')
display(corr_df.round(4))

if len(corr_df) >= 2:
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr_df.to_numpy(), cmap='YlGnBu', vmin=0.5, vmax=1.0)
    ax.set_xticks(range(len(corr_df.columns)))
    ax.set_xticklabels(corr_df.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(corr_df.index)))
    ax.set_yticklabels(corr_df.index)
    for i in range(len(corr_df)):
        for j in range(len(corr_df)):
            ax.text(j, i, f'{corr_df.iloc[i, j]:.3f}', ha='center', va='center', color='black')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title('Model Diversity: Log-odds Pearson Correlation')
    plt.tight_layout()
    plt.show()

### [해석 3] 단일 모델 성능 비교 및 예측 다양성(Diversity) 상관 분석
- 정형 파생 피처 기반 GBDT 모델군(`LightGBM`, `CatBoost`, `XGBoost`)과 시퀀스 어텐션 기반 신경망(`deep_seq_dcn`)의 `AP`, `50:50 WLL`, `Combined Score` 및 `OOT Score`를 동일 분할 위에서 나란히 비교했습니다.
- 이종 모델(`GBDT` vs `DeepCTR`) 간 `raw_logit` 상관계수가 동종 트리 모델 간 상관계수보다 낮을수록 결합 시 순위 변별력(`AP`)과 분산 감소(`WLL`) 시너지가 극대화됩니다.

In [ ]:
blender = OOFLogitBlender(
    objective=blend_cfg.get('objective', 'combined_score'),
    blend_space=blend_cfg.get('blend_space', 'logit'),
    opt_max_samples=blend_cfg.get('opt_max_samples', 500_000),
    n_restarts=int(blend_cfg.get('n_restarts', 24)),
    max_iter=int(blend_cfg.get('max_iter', 120)),
    test_agg=blend_cfg.get('test_agg', 'logit_mean'),
    seed=int(cfg.get('seed', 42)),
)
blend_res = blender.fit_blend(oof_dfs=oof_dfs, test_dfs=test_dfs, reference_test_ids=reference_ids)

fold_w_df = pd.DataFrame(blend_res.fold_weights)
print('=== Mean OOF Blending Weights ===')
for m_name, w_val in blend_res.weights.items():
    print(f'  {m_name:22s}: {w_val:.4f}')

display(fold_w_df)
display(pd.DataFrame(blend_res.fold_metrics))

best_single = single_df.iloc[0]
print(
    f'Best Single ({best_single["model"]}) -> AP: {best_single["ap"]:.6f}, '
    f'WLL: {best_single["wll"]:.6f}, Score: {best_single["score"]:.6f}'
)
print(
    f'OOF Logit Blend          -> AP: {blend_res.oof_metrics.ap:.6f}, '
    f'WLL: {blend_res.oof_metrics.wll:.6f}, Score: {blend_res.oof_metrics.score:.6f}'
)

### [해석 4] Out-of-Fold Log-odds 블렌딩 가중치 탐색 및 다양성 이득 확인
- 각 폴드 `k`마다 오직 `fold != k` 학습 데이터로만 비음수 가중치(`w >= 0, sum(w) = 1`)를 최적화하고 `fold == k`에 적용해 가중치 탐색 과적합 없는 순수 OOF 블렌딩 성능을 산출했습니다.
- 폴드별 가중치 편차가 안정적인지 확인하고, 단일 최고 모델 대비 OOF `AP` 및 `Combined Score` 상승 폭을 검증합니다.

In [ ]:
blended_oof_logits = blend_res.oof_df['raw_logit'].to_numpy(dtype=np.float64)
pre_cal_probs = blend_res.oof_df['pred_prob'].to_numpy(dtype=np.float64)
pre_cal_eval = evaluate_predictions(y_all, pre_cal_probs)

oof_affine_cal = OutOfFoldCalibrator(
    l2_reg=float(cal_cfg.get('l2_reg', 1.0e-6)),
    min_scale=float(cal_cfg.get('min_scale', 1.0e-3)),
    max_scale=float(cal_cfg.get('max_scale', 20.0)),
    fit_intercept=bool(cal_cfg.get('fit_intercept', True)),
    test_agg=cal_cfg.get('test_agg', 'logit_mean'),
)
cal_res = oof_affine_cal.fit_transform_result(blended_oof_logits, y_all, fold_ids, input_space='logit')
post_cal_eval = evaluate_predictions(y_all, cal_res.oof_probs)

ap_diff = abs(post_cal_eval.ap - pre_cal_eval.ap)
for f_k in sorted(np.unique(fold_ids).tolist()):
    m_k = fold_ids == f_k
    if np.unique(y_all[m_k]).size == 2:
        ap_k_pre = evaluate_predictions(y_all[m_k], pre_cal_probs[m_k]).ap
        ap_k_post = evaluate_predictions(y_all[m_k], cal_res.oof_probs[m_k]).ap
        assert abs(ap_k_post - ap_k_pre) < 1e-10, f'Fold {f_k} 단조 보정 AP 변동: {ap_k_post - ap_k_pre}'
inf_ap_eval = evaluate_predictions(y_all, oof_affine_cal.get_inference_calibrator().predict_proba(blended_oof_logits))
assert abs(inf_ap_eval.ap - pre_cal_eval.ap) < 1e-10, '평균 Affine 보정기 AP 변동 감지'

print(f'Mean Affine Params -> scale(a): {cal_res.mean_scale:.6f}, bias(b): {cal_res.mean_bias:.6f}')
print(f'Pre-Calibration   -> AP: {pre_cal_eval.ap:.6f}, WLL: {pre_cal_eval.wll:.6f}, Score: {pre_cal_eval.score:.6f}')
print(f'Post-Calibration  -> AP: {post_cal_eval.ap:.6f}, WLL: {post_cal_eval.wll:.6f}, Score: {post_cal_eval.score:.6f} (|dAP|={ap_diff:.2e})')
display(pd.DataFrame(cal_res.fold_params))

# 50:50 가중치 기준 Reliability Curve(보정 곡선) 계산 및 시각화
def compute_weighted_reliability_bins(y_true, y_prob, n_bins=10):
    n_pos = max(int(np.sum(y_true == 1)), 1)
    n_neg = max(int(np.sum(y_true == 0)), 1)
    w = np.where(y_true == 1, 0.5 / n_pos, 0.5 / n_neg)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        mask = (y_prob >= lo) & (y_prob <= hi if b == n_bins - 1 else y_prob < hi)
        if not np.any(mask) or np.sum(w[mask]) <= 0:
            continue
        w_b = w[mask]
        rows.append({
            'bin': f'{lo:.1f}-{hi:.1f}',
            'mean_pred': float(np.average(y_prob[mask], weights=w_b)),
            'weighted_emp_ctr': float(np.average(y_true[mask], weights=w_b)),
            'count': int(np.sum(mask)),
        })
    return pd.DataFrame(rows)

rel_before = compute_weighted_reliability_bins(y_all, pre_cal_probs)
rel_after = compute_weighted_reliability_bins(y_all, cal_res.oof_probs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect 50:50 Calibration')
axes[0].plot(rel_before['mean_pred'], rel_before['weighted_emp_ctr'], 'o-', label='Before Calibration')
axes[0].plot(rel_after['mean_pred'], rel_after['weighted_emp_ctr'], 's-', label='After Affine OOF Cal')
axes[0].set_title('50:50 Weighted Reliability Curve')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('50:50 Weighted Empirical Positive Rate')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(pre_cal_probs, bins=40, alpha=0.5, density=True, label='Before')
axes[1].hist(cal_res.oof_probs, bins=40, alpha=0.5, density=True, label='After Affine OOF Cal')
axes[1].set_title('OOF Predicted Probability Distribution')
axes[1].set_xlabel('Predicted Probability')
axes[1].legend()
plt.tight_layout()
plt.show()

### [해석 5] Out-of-Fold `50:50 WLL` 확률 보정(`AP` 보존 검증 & Reliability Curve)
- `AffineLogitCalibrator`의 양수 기울기(`a > 0`) 제약 덕분에 **순위 기반 `AP`가 소수점 오차 없이(`|dAP| < 1e-10`) 100% 보존**되었으며, OOF 기준으로 `50:50 WLL`과 최종 `Score`가 정밀 보정되었음을 확인했습니다.
- 왼쪽 50:50 가중 보정 곡선(Reliability Curve)에서 대각선(`y = x`)과의 밀착도가 높아질수록 `pos_logloss`와 `neg_logloss` 간 균형이 이상적으로 수렴합니다.

In [ ]:
train_tab_path = Path(paths_cfg['train_tabular'])
test_tab_path = Path(paths_cfg['test_tabular'])
gb_cfg = cal_cfg.get('group_bias', {})
group_col = gb_cfg.get('group_col', 'inventory_id')

active_calibrator = oof_affine_cal
final_oof_probs = cal_res.oof_probs
final_oof_eval = post_cal_eval
selected_cal_name = 'affine_logit'
tr_groups = None
te_groups = None

if train_tab_path.exists():
    schema_names = set(pq.ParquetFile(train_tab_path).schema_arrow.names)
    meta_cols = [c for c in ('row_id', 'inventory_id', 'day_of_week', 'hour') if c in schema_names]
    meta_df = pq.read_table(train_tab_path, columns=meta_cols).to_pandas()
    if 'row_id' in meta_df.columns:
        meta_df = meta_df.set_index('row_id').reindex(row_ids).reset_index()

    # 주요 지면(Top 10 inventory_id) 및 요일별 슬라이스 WLL 점검
    if 'inventory_id' in meta_df.columns:
        tr_groups = meta_df['inventory_id'].to_numpy()
        top_invs = meta_df['inventory_id'].value_counts().head(10).index.tolist()
        slice_rows = []
        for inv in top_invs:
            m = (meta_df['inventory_id'] == inv).to_numpy()
            if np.unique(y_all[m]).size < 2:
                continue
            wll_g, _, _ = weighted_logloss_50_50(y_all[m], cal_res.oof_probs[m])
            slice_rows.append({
                'inventory_id': inv,
                'rows': int(m.sum()),
                'ctr': float(y_all[m].mean()),
                'pred_mean': float(cal_res.oof_probs[m].mean()),
                'slice_wll': wll_g,
            })
        print('=== Top-10 Inventory Slice Calibration ===')
        display(pd.DataFrame(slice_rows))

        # GroupBiasShrinkageCalibrator OOF 비교 실험
        gb_oof_cal = OutOfFoldCalibrator(
            calibrator_factory=lambda: GroupBiasShrinkageCalibrator(
                shrinkage=float(gb_cfg.get('shrinkage', 500.0)),
                max_offset=float(gb_cfg.get('max_offset', 3.0)),
                l2_reg=float(cal_cfg.get('l2_reg', 1.0e-6)),
            ),
            test_agg=cal_cfg.get('test_agg', 'logit_mean'),
        )
        gb_probs = gb_oof_cal.fit_transform(
            blended_oof_logits, y_all, fold_ids, groups=tr_groups, input_space='logit'
        )
        gb_eval = evaluate_predictions(y_all, gb_probs)
        print(
            f'GroupBiasShrinkage OOF -> AP: {gb_eval.ap:.6f}, '
            f'WLL: {gb_eval.wll:.6f}, Score: {gb_eval.score:.6f}'
        )

        if (bool(gb_cfg.get('enabled', False)) or str(cal_cfg.get('method', '')) == 'group_shrinkage') and (
            str(cal_cfg.get('method', '')) == 'group_shrinkage'
            or (not bool(gb_cfg.get('only_accept_if_improves', True)))
            or (gb_eval.score > post_cal_eval.score + 1e-6)
        ):
            active_calibrator = gb_oof_cal
            final_oof_probs = gb_probs
            final_oof_eval = gb_eval
            selected_cal_name = 'group_shrinkage'
            if test_tab_path.exists():
                te_schema_names = set(pq.ParquetFile(test_tab_path).schema_arrow.names)
                if id_col not in te_schema_names or group_col not in te_schema_names:
                    raise KeyError(f'test_tabular에 {id_col} 및 {group_col} 컬럼이 모두 필요합니다.')
                te_meta_df = pq.read_table(test_tab_path, columns=[id_col, group_col]).to_pandas()
                expected_te_ids = reference_ids if reference_ids is not None else blend_res.test_df[id_col].to_numpy()
                if not _id_arrays_equal(te_meta_df[id_col].to_numpy(), expected_te_ids):
                    raise ValueError(f'test_tabular의 {id_col} 순서가 reference_ids / test 예측과 일치하지 않습니다.')
                te_groups = te_meta_df[group_col].to_numpy()

    del meta_df
    gc.collect()

print(f'Selected Final Calibrator: {selected_cal_name} (OOF Score={final_oof_eval.score:.6f})')

### [해석 6] 지면(`inventory_id`)·시간대별 슬라이스 캘리브레이션 진단 및 Shrinkage 보정 비교
- `train_tabular_diet.parquet`에서 전체 데이터 대신 메타 컬럼만 선택 로드해 상위 지면(`inventory_id`)별 실제 CTR 대비 예측 확률 편향과 슬라이스 WLL을 점검했습니다.
- `GroupBiasShrinkageCalibrator`는 지면별 잔차 편향을 Empirical Bayes shrinkage `n / (n + shrinkage)`로 보정하나 지면 간 상대 점수가 미세 이동해 `AP`가 변할 수 있으므로, OOF `Combined Score`가 유의하게 개선될 때만 채택하도록 안전 게이트를 적용했습니다.

In [ ]:
test_blended_logits = blend_res.test_df['raw_logit'].to_numpy(dtype=np.float64)
test_ids = blend_res.test_df[id_col].to_numpy()

if selected_cal_name == 'group_shrinkage':
    if te_groups is None:
        raise ValueError('group_shrinkage가 선택되었으나 test_tabular에서 te_groups를 로드할 수 없습니다.')
    final_test_probs = active_calibrator.predict_proba(
        test_blended_logits, groups=te_groups, input_space='logit'
    )
else:
    final_test_probs = oof_affine_cal.predict_proba(test_blended_logits, input_space='logit')

submission_path = Path(paths_cfg['submission_path'])
sub_df = generate_submission(
    test_ids=test_ids,
    pred_probs=final_test_probs,
    output_path=submission_path,
    id_col=id_col,
    target_col=sub_cfg.get('target_col', 'clicked'),
    eps=float(sub_cfg.get('eps', 1.0e-15)),
)

sub_check_df = sub_df if (sample_ids_df is None or sub_df[id_col].dtype == sample_ids_df[id_col].dtype) else sub_df.assign(**{id_col: sub_df[id_col].astype(str)})
verif_report = verify_submission_df(
    sub_df=sub_check_df,
    sample_sub_df=sample_ids_df,
    expected_rows=int(sub_cfg['expected_rows']) if sub_cfg.get('expected_rows') is not None else None,
    id_col=id_col,
    target_col=sub_cfg.get('target_col', 'clicked'),
    raise_on_error=True,
)
print('=== Submission Integrity Report ===')
print(json.dumps(verif_report, ensure_ascii=False, indent=2))
display(sub_df.head(10))

# 보정기 및 리포트 저장
calibrator_path = Path(paths_cfg['calibrator_path'])
calibrator_path.parent.mkdir(parents=True, exist_ok=True)
with calibrator_path.open('w', encoding='utf-8') as fp:
    json.dump(
        {
            'weights': blend_res.weights,
            'selected_calibration_method': selected_cal_name,
            'oof_calibrator': active_calibrator.to_dict(),
            'inference_affine_calibrator': oof_affine_cal.get_inference_calibrator().to_dict(),
        },
        fp,
        ensure_ascii=False,
        indent=2,
    )

report_path = Path(paths_cfg['report_path'])
report_path.parent.mkdir(parents=True, exist_ok=True)
with report_path.open('w', encoding='utf-8') as fp:
    json.dump(
        {
            'models': blend_res.model_names,
            'weights': blend_res.weights,
            'single_model_metrics': blend_res.single_model_metrics,
            'pre_calibration_oof': pre_cal_eval.to_dict(),
            'final_oof': final_oof_eval.to_dict(),
            'selected_calibration_method': selected_cal_name,
            'submission_verification': verif_report,
        },
        fp,
        ensure_ascii=False,
        indent=2,
    )

# 실시간 추론 레이어(RealtimeCTRPredictor) 스모크 검증
inf_calibrator = oof_affine_cal.get_inference_calibrator()
demo_store = OfflineFeatureStore().register_table(
    'inventory_id',
    pd.DataFrame({'inventory_id': [1, 2], 'inv_offline_ctr_prior': [0.025, 0.018]}),
)
demo_predictors = {
    m: (lambda df, val=float(test_blended_logits[0]): np.full(len(df), val, dtype=np.float64))
    for m in blend_res.model_names
}
rt_predictor = RealtimeCTRPredictor(
    model_predictors=demo_predictors,
    weights=blend_res.weights,
    calibrator=inf_calibrator,
    offline_store=demo_store,
    model_cadences={m: ('nearline_gpu_or_async' if 'deep' in m else 'realtime_sync') for m in blend_res.model_names},
)
rt_out = rt_predictor.predict_one({'ID': 'DEMO_REQ_001', 'inventory_id': 1, 'hour': 14})
print('RealtimeCTRPredictor smoke output:', rt_out)

### [해석 7] 최종 제출 파일 무결성 검증, 아티팩트 저장 및 실시간 서빙 스모크 점검
- 최종 제출 파일(`submission_ensemble_wll50.csv`)이 **1,527,298행**, `["ID", "clicked"]` 컬럼, `sample_submission.csv`와 **ID 순서 100% 일치**, `NaN/Inf` **0건**, **`(0, 1)` 개구간** 조건을 모두 통과했음을 확인했습니다.
- 블렌딩 가중치·보정 파라미터 JSON(`artifacts/ensemble_wll50_calibrator.json`)과 평가 리포트(`reports/ensemble_wll50_report.json`)를 저장하고, `RealtimeCTRPredictor`로 단건 실시간 추론 지연시간 및 출력 일관성을 검증했습니다.

### [종합 요약 및 본선 코드 검증·서빙 체크리스트]
1. **최종 앙상블 및 확률 보정 구조**:
   - 정형 FE 기반 다중 GBDT(`LightGBM`, `CatBoost`, `XGBoost`) + 시퀀스 딥러닝(`SeqDCNv2CTRModel`)의 `raw_logit`을 5-Fold Out-of-Fold Log-odds 블렌딩으로 결합.
   - 결합된 Logit에 `OutOfFoldCalibrator(AffineLogitCalibrator)`를 적용해 `AP` 100% 불변 및 `50:50 WLL` 정밀 보정 달성.
2. **본선 코드 검증 대비 재현 CLI 파이프라인 (`scripts/`)**:
   - Step 1: `python scripts/run_diet_and_cv.py --config configs/data_diet.yaml --cv-config configs/cv_strategy.yaml`
   - Step 2: `python scripts/extract_seq_features.py --config configs/deep_seq_dcn.yaml`
   - Step 3a: `python scripts/train_gbdt.py --config configs/gbdt_fe_tuned.yaml --model all`
   - Step 3b: `python scripts/train_deepctr.py --config configs/deep_seq_dcn.yaml`
   - Step 4: `python scripts/blend_and_submit.py --config configs/ensemble_wll50.yaml`
   - Step 5: `python scripts/verify_submission.py --config configs/ensemble_wll50.yaml`
3. **실시간 서빙 아키텍처 (`RealtimeCTRPredictor`)**:
   - **오프라인 배치 레이어 (`OfflineFeatureStore`)**: 주기적으로 집계된 `history_a`/유저 시퀀스 요약·지면/광고셋 통계를 O(1) 조회 테이블로 서빙.
   - **온라인 컨텍스트 + 다중 추론 주기 결합**: 요청 시점의 시간/지면 교호작용 피처를 계산하고, 실시간 GBDT 로짓과 비동기/GPU DeepCTR 로짓을 Log-odds 가중 결합 후 경량 `AffineLogitCalibrator` 1회 연산으로 즉시 보정 확률을 반환합니다.